# Auditing Synthea medications

Synthea is the reference open-source synthetic EHR generator. Its medications table poses a natural-seeming cost prediction task with a fatal property: the cost is exact bookkeeping.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd() if (Path.cwd() / 'datasets').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from synthaudit import Audit
df = pd.read_csv(ROOT / 'datasets' / 'synthea_medications.csv.gz')
audit = Audit(df, target='TOTALCOST', name='synthea_medications')
results = audit.run()

[synthaudit] auditing 'synthea_medications' (42,989 rows x 13 cols, target=TOTALCOST)


[synthaudit] profile: 6 numeric, 1 categorical, 0 duplicate rows


[synthaudit] identity mining: 2 exact, 0 near identities, 2 FDs, 0 rule-derived labels


[synthaudit] determinism sweep: 0 deterministic, 0 near-deterministic columns


[synthaudit] causal scan: 4 edges on stochastic core (1 columns excluded)


[synthaudit] leakage audit: 3 critical, 0 high findings


[synthaudit] BTI = 0.242 (grade F) pillars={'L': 0.0, 'F': 0.8333, 'H': 1.0, 'R': 1.0, 'I': 1.0}


[synthaudit] done in 11.17s


In [2]:
for f in results['identity']['identities']:
    print(f['type'], '|', f.get('equation'), '| disposition:', f.get('disposition'))
for fd in results['identity']['functional_dependencies']:
    print('FD:', fd['equation'], '| g3 =', fd['g3_violation_rate'])

power_law | DISPENSES = 1 * BASE_COST^-1 * TOTALCOST^1 | disposition: target_leakage
FD: CODE -> DESCRIPTION | g3 = 0.002117
FD: REASONCODE -> REASONDESCRIPTION | g3 = 0.0


Verify the recovered identity against the raw file, no model involved. This is what *constructive* evidence means: anyone can check it.

In [3]:
resid = (df['TOTALCOST'] - df['BASE_COST'] * df['DISPENSES']).abs()
print('max |TOTALCOST - BASE_COST*DISPENSES| =', float(resid.max()))

max |TOTALCOST - BASE_COST*DISPENSES| = 5.820766091346741e-11


In [4]:
print('grade:', results['scoring']['grade'], '| L pillar:', results['scoring']['pillars']['L'])

grade: F | L pillar: 0.0


A subtlety worth knowing: a gradient-boosted probe reaches only mediocre R-squared on this target (heavy-tailed costs defeat tree extrapolation), so a predictability screen alone would under-detect this leak. The identity miner proves exact recoverability regardless, and the verdict is seed-invariant. Bookkeeping is legitimate accounting AND a broken benchmark task: disposition and grade capture both halves of that sentence.